<a href="https://colab.research.google.com/github/anuj-walia/MyGenAIExperiments/blob/main/BasicEmbeddingSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 70.3 MB/s eta 0:00:00


In [2]:
!uv

An extremely fast Python package manager.

Usage: uv [OPTIONS] <COMMAND>

Commands:
  auth     Manage authentication
  run      Run a command or script
  init     Create a new project
  add      Add dependencies to the project
  remove   Remove dependencies from the project
  version  Read or update the project's version
  sync     Update the project's environment
  lock     Update the project's lockfile
  export   Export the project's lockfile to an alternate format
  tree     Display the project's dependency tree
  format   Format Python code in the project
  tool     Run and install commands provided by Python packages
  python   Manage Python versions and installations
  pip      Manage Python packages with a pip-compatible interface
  venv     Create a virtual environment
  build    Build Python packages into source distributions and wheels
  publish  Upload distributions to an index
  cache    Manage uv's cache
  self     Manage the uv executable
  help     Display documentation 

In [3]:
!uv pip install sentence-transformers

Using Python 3.12.11 environment at: /usr
Audited 1 package in 187ms


In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained model.
# 'all-MiniLM-L6-v2' is a popular, fast, and effective model.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Let's use the 'chunks' list we created in the last step.
# If you are starting fresh, you can uncomment and use this example:
chunks = [
    "Be as pessemistic as you can.",
    "Everything in life is negative and is toi screw you.",
    "You cannot control consequences but learn to enjoy them.",
    "Always think of positive outcomes."
]

# Generate the embeddings for our chunks
embeddings = model.encode(chunks)

# Let's check the shape of our result. It will be
# (number of chunks, dimension of the embedding)
print(f"Shape of our embeddings: {embeddings.shape}")

# To see one, we can print the first embedding
# print(f"\nFirst embedding vector: {embeddings[0]}")

Shape of our embeddings: (4, 384)


In [14]:
print(type(embeddings))

<class 'numpy.ndarray'>


In [8]:
embeddings[0]

array([-2.58423928e-02,  4.75612879e-02,  1.59470607e-02,  2.84728617e-03,
        2.47500688e-02, -5.93983792e-02,  8.36752206e-02,  1.49444342e-02,
       -7.58973360e-02, -8.30548555e-02, -3.96877564e-02, -6.67489767e-02,
       -3.33394296e-02,  4.79050875e-02, -5.60800247e-02, -5.84347397e-02,
       -1.05139976e-02,  3.50247733e-02,  1.95245235e-03, -8.07388723e-02,
       -8.46934156e-04, -1.10364472e-02,  1.84183586e-02,  2.91559324e-02,
       -2.54233554e-02,  7.36677051e-02, -1.09336134e-02, -1.28935743e-02,
        1.54687045e-02,  4.40959185e-02,  1.66461628e-03,  2.23746952e-02,
        7.98375458e-02, -5.27389906e-02, -4.21886593e-02, -5.62878326e-02,
        6.93481462e-03, -2.49429271e-02, -1.10268638e-01, -3.70393656e-02,
       -2.10836828e-02, -6.34343177e-02,  1.83410496e-02,  6.28748760e-02,
       -1.62961632e-02, -1.41726732e-02, -6.26229495e-02,  2.78080441e-02,
        2.34763138e-02,  1.07508071e-01, -1.87619980e-02, -8.34963843e-02,
        2.51795426e-02, -

In [18]:
from sentence_transformers import util
import torch

def search(query):
    # 1. Encode the query to get its vector.
    query_embedding = model.encode(query)
    print(f"query embedding shape{query_embedding.shape}")
    print(query_embedding[0])

    # 2. Calculate cosine similarity between the query and all chunk vectors.
    cos_scores = util.cos_sim(query_embedding, embeddings)
    print(type(cos_scores))
    print(cos_scores)
    print(len(cos_scores))

    # 3. Find the top 3 most similar chunks.
    top_results = torch.topk(cos_scores[0], k=4)
    print(top_results)

    print(f"Query: {query}\n")
    print("Top 3 most similar chunks in our documents:")

    for score, idx in zip(top_results[0], top_results[1]):
        print(f"- {chunks[idx]} (Score: {score:.4f})")

# --- Let's try it out! ---
# Using the example chunks from before:
# chunks = [
#     "My application helps users track their daily water intake.",
#     "To add a new entry, press the blue plus button on the main screen.",
#     "You can view your history by navigating to the 'History' tab in the menu."
# ]

search("How do I see positive things in life?")

query embedding shape(384,)
0.038460895
<class 'torch.Tensor'>
tensor([[0.3176, 0.4951, 0.2196, 0.6016]])
1
torch.return_types.topk(
values=tensor([0.6016, 0.4951, 0.3176, 0.2196]),
indices=tensor([3, 1, 0, 2]))
Query: How do I see positive things in life?

Top 3 most similar chunks in our documents:
- Always think of positive outcomes. (Score: 0.6016)
- Everything in life is negative and is toi screw you. (Score: 0.4951)
- Be as pessemistic as you can. (Score: 0.3176)
- You cannot control consequences but learn to enjoy them. (Score: 0.2196)


In [19]:
chunks = [
    "Cat.",
    "Mouse.",
    "Sparrow.",
    "lion."
]

# Generate the embeddings for our chunks
embeddings = model.encode(chunks)


In [20]:
search("Cat")

query embedding shape(384,)
0.03733034
<class 'torch.Tensor'>
tensor([[0.8794, 0.4004, 0.3689, 0.5601]])
1
torch.return_types.topk(
values=tensor([0.8794, 0.5601, 0.4004, 0.3689]),
indices=tensor([0, 3, 1, 2]))
Query: Cat

Top 3 most similar chunks in our documents:
- Cat. (Score: 0.8794)
- lion. (Score: 0.5601)
- Mouse. (Score: 0.4004)
- Sparrow. (Score: 0.3689)


In [21]:
def create_prompt(query, retrieved_chunks):
    # The chunks are a list of strings, so we'll join them together
    context_str = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""
Use the following pieces of context to answer the question at the end.
If you don't know the answer from the context provided, just say that you don't know.

Context:
{context_str}

Question: {query}

Answer:
"""
    return prompt

# Let's imagine our search function returned these top 2 chunks:
top_chunks = [
    "You can view your history by navigating to the 'History' tab in the menu.",
    "The history screen shows a log of all past entries."
]
user_question = "How do I see my history?"

# Create the final prompt
final_prompt = create_prompt(user_question, top_chunks)

print(final_prompt)


Use the following pieces of context to answer the question at the end.
If you don't know the answer from the context provided, just say that you don't know.

Context:
You can view your history by navigating to the 'History' tab in the menu.

---

The history screen shows a log of all past entries.

Question: How do I see my history?

Answer:



In [23]:
!uv pip install transformers torch

Using Python 3.12.11 environment at: /usr
Audited 2 packages in 473ms


In [24]:
!uv pip install transformers

Using Python 3.12.11 environment at: /usr
Audited 1 package in 514ms


In [25]:
# You would need to install the libraries first:
# pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch

# Load the tokenizer and model directly from the Hugging Face Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Our list of text chunks
chunks = [
    "You can view your history by navigating to the 'History' tab in the menu.",
    "The history screen shows a log of all past entries."
]

# 1. Tokenize the sentences
encoded_input = tokenizer(chunks, padding=True, truncation=True, return_tensors='pt')

# 2. Pass tokens through the model to get token embeddings
with torch.no_grad(): # This is for efficiency
    model_output = model(**encoded_input)

token_embeddings = model_output.last_hidden_state

# 3. Perform mean pooling to get one vector per sentence
# We average the token embeddings, using the attention mask to ignore padding tokens.
attention_mask = encoded_input['attention_mask']
input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
sentence_embeddings = sum_embeddings / sum_mask

print("Shape of our final sentence embeddings:", sentence_embeddings.shape)

Shape of our final sentence embeddings: torch.Size([2, 384])
